In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import wandb

# export WANDB_API_KEY=7855589e123f4caee4595a43d22c86cce82fbda1
wandb.login(key="7855589e123f4caee4595a43d22c86cce82fbda1",relogin=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\mza0288\_netrc
wandb: Currently logged in as: mmabappy1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from ultralytics import YOLO
from ultralytics import RTDETR
import torch
import os
import pandas as pd

models = ["rtdetr-l.pt", "rtdetr-x.pt"]
data_path = "C:/data/data_set/data.yaml"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

epochs = 200
run_name = "None"

for model_name in models:
    print(f"\nTraining model: {model_name}")

    batch_size = 8
    imgsz = 640

    # Clear CUDA cache before starting new training (if using CUDA)
    if device == "cuda":
        torch.cuda.empty_cache()

    model = RTDETR(model_name).to(device)

    run_name = f"{model_name.split('.')[0]}-bs{batch_size}-ep{epochs}"

    results = model.train(
        data=data_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=imgsz,
        device=device,
        seed=7,
        optimizer="SGD",
        project="runs/poultry",
        name=run_name,
        save=True,
       verbose=True,
    )

    # Initialize wandb before training (your existing init)
    wandb.init(project="poultry", name=run_name)
    
    
    # Path to training metrics CSV created by ultralytics:
    # Usually: runs/train/<run_name>/metrics.csv
    metrics_csv_path = f"runs/poultry/{run_name}/results.csv"

    # Load the CSV into DataFrame
    df = pd.read_csv(metrics_csv_path)
    
    # Iterate over each epoch's metrics row
    for _, row in df.iterrows():
        # Convert epoch to int for clarity
        epoch_num = int(row['epoch'])
    
        # Prepare dictionary of metrics, exclude epoch key
        metrics = {k: float(v) for k, v in row.items() if k != 'epoch'}
    
        # Add epoch number to the dictionary for logging
        metrics['epoch'] = epoch_num
    
        # Log metrics for this epoch to wandb
        wandb.log(metrics)
    
    wandb.finish()

Using device: cuda

Training model: rtdetr-l.pt


100%|██████████| 63.4M/63.4M [00:05<00:00, 11.8MB/s]


New https://pypi.org/project/ultralytics/8.3.190 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.171  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/data/data_set/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=rtdetr-l-bs8-ep200, nbs=64, nms=False, opse

train: Scanning C:\data\data_set\train\labels.cache... 4824 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4824/4824 [00:00<?, ?it/s]Scanning C:\data\data_set\train\labels.cache... 4824 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4824/4824 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.00.0 ms, read: 271.856.2 MB/s, size: 1755.9 KB)


...

  0%|          | 0/603 [00:00<?, ?it/s]c:\Users\mza0288\AppData\Local\anaconda3\envs\torchenv\Lib\site-packages\torch\autograd\graph.py:823: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:96.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    200/200      9.66G     0.1595     0.2518    0.02145         36        640: 100%|██████████| 603/603 [02:18<00:00,  4.36it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:05<00:00,  8.71it/s]


                   all        702       3143      0.954      0.969      0.953      0.739

200 epochs completed in 8.064 hours.
Optimizer stripped from runs\poultry\rtdetr-x-bs8-ep200\weights\last.pt, 135.4MB
Optimizer stripped from runs\poultry\rtdetr-x-bs8-ep200\weights\best.pt, 135.4MB

Validating runs\poultry\rtdetr-x-bs8-ep200\weights\best.pt...
Ultralytics 8.3.171  Python-3.12.9 torch-2.6.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
rt-detr-x summary: 370 layers, 65,471,546 parameters, 0 gradients, 222.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:05<00:00,  8.09it/s]


                   all        702       3143      0.963      0.965      0.958      0.751
       broiler breeder         68        193      0.945      0.948      0.932      0.658
         broiler house        645       2950      0.981      0.982      0.984      0.844
Speed: 0.1ms preprocess, 4.4ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs\poultry\rtdetr-x-bs8-ep200


epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇██
lr/pg0,█▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr/pg1,██▇▇▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▁▁▁
lr/pg2,▆████▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▁▁
metrics/mAP50(B),▁▆▆▇▇█████████████████████████▇▇▇███████
metrics/mAP50-95(B),▁▁▆▆▆▆▇▇▇▇▇▇▇▇▇█▇▇██████████████████████
metrics/precision(B),▁▆▇▆█▇█▇█▇██████████████████████████████
metrics/recall(B),▁▂▄▇▇█▇█▇█▇█████████████████████████████
time,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇███
train/cls_loss,█▇▇▆▆▆▅▅▅▅▅▅▅▅▅▅▅▄▄▄▃▄▄▃▃▃▄▄▃▃▃▂▂▂▂▃▂▁▂▁
train/giou_loss,█▇▆▆▆▅▄▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▃▂▂▁▂▂▂▃▂▂▁▁▁
